# 01 — Cohort Feasibility Audit

## 1. Objective

This notebook evaluates the feasibility of a fixed-landmark clinical
prediction study using Health System A.

The proposed landmark is ICU hour 6. Clinical information available
through the landmark will be used to predict subsequent sepsis onset.

Several candidate prediction horizons are evaluated using System A only
before the primary study design is frozen:

- **Landmark time:** ICU hour 6
- **Feature window:** available clinical history up to and including ICU hour 6
- **Candidate horizon ends:** ICU hours 12, 18, and 24
- **Development system:** Health System A
- **External validation system:** Health System B, which is not examined
  in this notebook

No predictive models are trained here. The purpose is to verify data
structure, outcome reconstructability, follow-up completeness, cohort
size, and event frequency before the primary prediction task is frozen.

## 2. Data Loading and Structural Checks

Load all System A patient records and verify the expected sample size
and variable structure.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Resolve project root whether the notebook kernel starts from the
# repository root or from the notebooks/ directory.
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

A_DIR = ROOT / "data" / "raw" / "training" / "training_setA"

assert A_DIR.exists(), f"System A directory not found: {A_DIR}"

files_a = sorted(A_DIR.glob("*.psv"))

print("Project root: repository root")
print(f"System A directory: {A_DIR.relative_to(ROOT)}")
print(f"Patient files: {len(files_a):,}")

assert len(files_a) == 20_336, (
    f"Expected 20,336 System A files, found {len(files_a):,}"
)

Project root: repository root
System A directory: data/raw/training/training_setA
Patient files: 20,336


In [2]:
example = pd.read_csv(files_a[0], sep="|")

display(example.head())

print(f"Rows: {len(example)}")
print(f"Columns: {len(example.columns)}")
print(example.columns.tolist())

required_columns = {"ICULOS", "SepsisLabel"}

assert required_columns.issubset(example.columns)

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0


Rows: 54
Columns: 41
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']


## 3. Patient-Level Outcome Reconstruction

For septic patients, the released challenge label is shifted six hours
before the Sepsis-3 onset time.

When the first positive `SepsisLabel` is preceded by at least one
observed negative label, the transition is observed and the onset time
can be reconstructed as:

$$
t_{\mathrm{sepsis}}
=
t_{\mathrm{first\ positive\ label}} + 6
$$

If the patient's first available ICU row is already positive, the
0-to-1 transition occurred at or before the beginning of the observed
record. The exact onset time therefore cannot be recovered from the
released label. These cases are flagged as left-truncated rather than
assigned an artificial onset time.

This section constructs one audit record per patient. No prediction
features are created.

In [3]:
records = []

for path in files_a:
    df = pd.read_csv(path, sep="|")

    iculos = df["ICULOS"].to_numpy()
    labels = df["SepsisLabel"].to_numpy()

    # Basic structural checks
    min_iculos = int(np.min(iculos))
    max_iculos = int(np.max(iculos))
    n_rows = len(df)

    duplicate_iculos = len(np.unique(iculos)) != len(iculos)
    monotone_iculos = bool(np.all(np.diff(iculos) > 0))
    contiguous_iculos = bool(np.all(np.diff(iculos) == 1))
    starts_at_hour1 = min_iculos == 1

    valid_labels = bool(np.isin(labels, [0, 1]).all())

    # Under the released label definition, once SepsisLabel becomes 1
    # for a septic patient it should not revert to 0.
    monotone_labels = (
        bool(np.all(np.diff(labels) >= 0))
        if valid_labels
        else False
    )

    positive_hours = iculos[labels == 1]

    if len(positive_hours) == 0:
        septic = False
        first_positive = np.nan
        reconstructed_onset = np.nan
        onset_left_truncated = False

    else:
        septic = True
        first_positive = int(np.min(positive_hours))

        # Exact onset cannot be reconstructed if the first available
        # observation is already positive, regardless of whether the
        # record begins at ICULOS 1 or later.
        onset_left_truncated = bool(labels[0] == 1)

        if onset_left_truncated:
            reconstructed_onset = np.nan
        else:
            reconstructed_onset = first_positive + 6

    onset_beyond_record = (
        False
        if np.isnan(reconstructed_onset)
        else reconstructed_onset > max_iculos
    )

    records.append(
        {
            "patient_id": path.stem,
            "n_rows": n_rows,
            "min_iculos": min_iculos,
            "max_iculos": max_iculos,
            "starts_at_hour1": starts_at_hour1,
            "duplicate_iculos": duplicate_iculos,
            "monotone_iculos": monotone_iculos,
            "contiguous_iculos": contiguous_iculos,
            "valid_labels": valid_labels,
            "monotone_labels": monotone_labels,
            "septic": septic,
            "first_positive": first_positive,
            "reconstructed_onset": reconstructed_onset,
            "onset_left_truncated": onset_left_truncated,
            "onset_beyond_record": onset_beyond_record,
        }
    )

audit = pd.DataFrame(records)

audit.head()

,patient_id,n_rows,min_iculos,max_iculos,starts_at_hour1,duplicate_iculos,monotone_iculos,contiguous_iculos,valid_labels,monotone_labels,septic,first_positive,reconstructed_onset,onset_left_truncated,onset_beyond_record
0,p000001,54,1,54,True,False,True,True,True,True,False,NaN,NaN,False,False
1,p000002,23,1,23,True,False,True,True,True,True,False,NaN,NaN,False,False
2,p000003,48,1,48,True,False,True,True,True,True,False,NaN,NaN,False,False
3,p000004,29,1,29,True,False,True,True,True,True,False,NaN,NaN,False,False
4,p000005,48,2,49,False,False,True,True,True,True,False,NaN,NaN,False,False


## 4. Structural Data Audit

Before defining the analysis cohort, verify that ICU time indices and
outcome labels have the expected structure.

In [4]:
structural_summary = pd.Series(
    {
        "patients": len(audit),
        "starts_after_hour1": int(
            (~audit["starts_at_hour1"]).sum()
        ),
        "starts_after_hour6": int(
            (audit["min_iculos"] > 6).sum()
        ),
        "duplicate_ICULOS": int(
            audit["duplicate_iculos"].sum()
        ),
        "nonmonotone_ICULOS": int(
            (~audit["monotone_iculos"]).sum()
        ),
        "noncontiguous_ICULOS": int(
            (~audit["contiguous_iculos"]).sum()
        ),
        "invalid_SepsisLabel": int(
            (~audit["valid_labels"]).sum()
        ),
        "label_reversion": int(
            (~audit["monotone_labels"]).sum()
        ),
        "ever_septic": int(
            audit["septic"].sum()
        ),
        "onset_left_truncated": int(
            audit["onset_left_truncated"].sum()
        ),
        "reconstructed_onset_beyond_record": int(
            audit["onset_beyond_record"].sum()
        ),
    },
    name="System A",
)

structural_summary.to_frame()

,System A
patients,20336
starts_after_hour1,7497
starts_after_hour6,483
duplicate_ICULOS,0
nonmonotone_ICULOS,0
noncontiguous_ICULOS,0
invalid_SepsisLabel,0
label_reversion,0
ever_septic,1790
onset_left_truncated,203


In [5]:
landmark_coverage = pd.Series(
    {
        "contains_hour_6": int(
            (
                (audit["min_iculos"] <= 6)
                & (audit["max_iculos"] >= 6)
            ).sum()
        ),
        "contains_hour_12": int(
            (
                (audit["min_iculos"] <= 12)
                & (audit["max_iculos"] >= 12)
            ).sum()
        ),
        "contains_hour_18": int(
            (
                (audit["min_iculos"] <= 18)
                & (audit["max_iculos"] >= 18)
            ).sum()
        ),
        "contains_hour_24": int(
            (
                (audit["min_iculos"] <= 24)
                & (audit["max_iculos"] >= 24)
            ).sum()
        ),
    },
    name="patients",
)

landmark_coverage.to_frame()

,patients
contains_hour_6,19853
contains_hour_12,19918
contains_hour_18,18878
contains_hour_24,16240


In [6]:
audit["min_iculos"].value_counts().sort_index().to_frame("patients")

,patients
min_iculos,
1,12839
2,3388
3,1678
4,1003
5,576
6,369
7,208
8,109
9,60


In [7]:
audit.loc[
    audit["onset_beyond_record"],
    [
        "patient_id",
        "min_iculos",
        "max_iculos",
        "first_positive",
        "reconstructed_onset",
    ],
]

,patient_id,min_iculos,max_iculos,first_positive,reconstructed_onset
5692,p005693,2,70,65.0,71.0
18468,p018469,1,336,331.0,337.0


### Reconstructed Onset Beyond the Recorded ICU Window

Two patients have reconstructed sepsis onset one hour after the final
available ICU observation.

This is compatible with the shifted challenge labeling scheme: a
patient may enter the positive `SepsisLabel` period several hours before
the reconstructed clinical onset, while the released ICU record ends
during that early-warning interval.

These cases are therefore retained rather than treated as data errors.
For a fixed prediction horizon, they may contribute as negative cases
when their observed record extends through that horizon and the
reconstructed onset occurs later.

In [8]:
audit["max_iculos"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    20336.000000
mean        39.774194
std         22.552482
min          8.000000
10%         19.000000
25%         26.000000
50%         40.000000
75%         48.000000
90%         55.000000
95%         58.000000
99%        134.000000
max        336.000000
Name: max_iculos, dtype: float64

## 5. Candidate Landmark Cohorts

The landmark is fixed at:

$$
t_0 = 6\ \text{ICU hours}.
$$

Candidate prediction windows end at ICU hours 12, 18, and 24.

For each candidate horizon:

- the observed patient record must contain ICU hour 6;
- all available clinical history up to and including ICU hour 6 may be
  used for predictor construction;
- patients with left-truncated sepsis labels are excluded because their
  exact onset time cannot be reconstructed;
- patients with sepsis onset at or before the landmark are excluded
  because they are no longer at risk for incident sepsis;
- patients with reconstructed sepsis onset within the prediction window
  are classified as positive;
- patients without an event in the prediction window must have observed
  follow-up through the end of that window before being classified as
  negative.

The same eligibility logic is applied to every candidate horizon.

In [9]:
LANDMARK = 6


def build_landmark_cohort(horizon_end):
    c = audit.copy()

    # The observed record must actually contain the landmark hour.
    # max_iculos >= LANDMARK alone is insufficient because some
    # patient records begin well after ICU hour 6.
    c["observed_at_landmark"] = (
        (c["min_iculos"] <= LANDMARK)
        & (c["max_iculos"] >= LANDMARK)
    )

    # Patients whose reconstructed onset occurs at or before the
    # landmark are not at risk for incident sepsis after the landmark.
    c["prevalent_sepsis"] = (
        c["reconstructed_onset"].notna()
        & (c["reconstructed_onset"] <= LANDMARK)
    )

    # Incident sepsis during the candidate prediction window.
    c["incident_sepsis_in_window"] = (
        c["reconstructed_onset"].notna()
        & (c["reconstructed_onset"] > LANDMARK)
        & (c["reconstructed_onset"] <= horizon_end)
    )

    # Complete observed follow-up through the candidate horizon.
    c["complete_followup_to_horizon"] = (
        c["max_iculos"] >= horizon_end
    )

    # Positive cases are observable once an incident event occurs
    # within the window. Patients without an event in the window
    # require observed follow-up through the horizon to establish
    # a negative outcome.
    c["outcome_observable"] = (
        c["incident_sepsis_in_window"]
        | c["complete_followup_to_horizon"]
    )

    c["eligible"] = (
        c["observed_at_landmark"]
        & (~c["onset_left_truncated"])
        & (~c["prevalent_sepsis"])
        & c["outcome_observable"]
    )

    c["outcome"] = (
        c["incident_sepsis_in_window"].astype(int)
    )

    return c

## 6. Feasibility Summary and Horizon Selection

Compare candidate prediction horizons using System A only.

The comparison is based on cohort retention, follow-up completeness,
and incident event frequency. No predictive performance is considered,
and Health System B remains unexamined.

### Reconstructed Onset Distribution

The distribution below includes only patients whose onset can be
reconstructed exactly. Therefore, the absence of events at or before
hour 6 should not be interpreted as evidence that early sepsis does not
occur; patients whose positive labeling began before the observed
record are excluded from exact onset reconstruction.

In [10]:
exact_onsets = audit.loc[
    audit["reconstructed_onset"].notna(),
    "reconstructed_onset",
]

onset_bins = pd.cut(
    exact_onsets,
    bins=[0, 6, 12, 18, 24, 48, np.inf],
    right=True,
    labels=[
        "≤6h",
        "6–12h",
        "12–18h",
        "18–24h",
        "24–48h",
        ">48h",
    ],
)

onset_bins.value_counts(sort=False).to_frame("patients")

,patients
reconstructed_onset,
≤6h,0
6–12h,180
12–18h,191
18–24h,143
24–48h,345
>48h,728


### Candidate Horizon Comparison

Before freezing the primary prediction window, compare candidate
horizons using Health System A only.

The comparison is based on cohort retention, follow-up completeness,
and incident event availability rather than predictive performance.

Health System B remains completely unexamined during this decision.

In [11]:
def evaluate_horizon(horizon_end):
    c = build_landmark_cohort(horizon_end)

    analysis = c.loc[c["eligible"]].copy()

    n_eligible = len(analysis)
    n_positive = int(analysis["outcome"].sum())
    n_negative = n_eligible - n_positive

    incomplete_followup_exclusions = int(
        (
            c["observed_at_landmark"]
            & (~c["onset_left_truncated"])
            & (~c["prevalent_sepsis"])
            & (~c["incident_sepsis_in_window"])
            & (~c["complete_followup_to_horizon"])
        ).sum()
    )

    return {
        "prediction_window": f"{LANDMARK}–{horizon_end}h",
        "eligible_patients": n_eligible,
        "positive_events": n_positive,
        "negative_patients": n_negative,
        "event_rate": (
            n_positive / n_eligible
            if n_eligible
            else np.nan
        ),
        "incomplete_followup_exclusions": (
            incomplete_followup_exclusions
        ),
    }


horizon_comparison = pd.DataFrame(
    [
        evaluate_horizon(12),
        evaluate_horizon(18),
        evaluate_horizon(24),
    ]
)

horizon_comparison

,prediction_window,eligible_patients,positive_events,negative_patients,event_rate,incomplete_followup_exclusions
0,6–12h,19520,180,19340,0.009221,133
1,6–18h,18699,371,18328,0.019841,954
2,6–24h,16274,510,15764,0.031338,3379


## 7. Design Decision

**Final decision: retain ICU hour 6 as the landmark and use a 12-hour
prediction horizon ending at ICU hour 18.**

The candidate 6–12 hour window retained the largest cohort but yielded
only 180 incident sepsis events, providing relatively limited event
availability for later calibration, uncertainty estimation,
recalibration, and subgroup analyses.

Extending the horizon to ICU hour 18 approximately doubled the number
of incident events while preserving the large majority of the eligible
cohort.

Extending further to ICU hour 24 produced additional events but required
substantially more exclusions because of incomplete follow-up, resulting
in a less favorable trade-off between event availability and cohort
retention.

The primary prediction task is therefore frozen as:

$$
t_0 = 6\ \text{ICU hours},
$$

with outcome:

$$
Y_i =
\mathbf{1}
\left(
6 < t_{\mathrm{sepsis},i} \le 18
\right).
$$

Predictors will use all clinical information available up to and
including ICU hour 6.

Thus, the primary study asks:

> Among ICU patients who are observable and still at risk at ICU hour 6,
> can clinical information available by that landmark predict sepsis
> onset during the subsequent 12 hours?

This design decision was made using Health System A cohort feasibility,
follow-up completeness, and event frequency only. No predictive model
performance and no Health System B outcomes were examined during
horizon selection.

The landmark, prediction horizon, and primary outcome definition are
considered frozen for subsequent analyses.